<!-- Day 1 - Stats and Probability -->

<!-- Day 1  - Stats and Probability-->

In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
titanic = sns.load_dataset("titanic")

In [ ]:
titanic.head(10)

In [ ]:
# Calculate basic statistics for the data

In [ ]:
titanic.describe()

### Manual mean/median/variance/std vs `.describe()`

Goal: reproduce `describe()`'s `age` stats by hand with NumPy, and see exactly where naive NumPy calls diverge (NaN handling + `ddof`).

In [ ]:
# Naive attempt: plain np.mean/median/var/std, default ddof=0, no NaN skipping
age = titanic["age"]

naive_mean = np.mean(age)
naive_median = np.median(age)
naive_var = np.var(age)      # ddof=0 -> population variance
naive_std = np.std(age)      # ddof=0 -> population std

print("mean:  ", naive_mean)
print("median:", naive_median)
print("var:   ", naive_var)
print("std:   ", naive_std)

Everything came out `nan` — `age` has missing values, and plain `np.mean`/`np.var`/`np.std`/`np.median` don't skip NaNs. Fix: use the `np.nan*` variants, and set `ddof=1` to match pandas' sample-variance convention.

In [ ]:
# Fixed: nan-aware + ddof=1 (sample variance/std, matching pandas default)
manual_mean = np.nanmean(age)
manual_median = np.nanmedian(age)
manual_var = np.nanvar(age, ddof=1)
manual_std = np.nanstd(age, ddof=1)

print("mean:  ", manual_mean)
print("median:", manual_median)
print("var:   ", manual_var)
print("std:   ", manual_std)

In [ ]:
# Verify against describe() and pandas' own .mean()/.median()/.var()/.std()
desc = titanic["age"].describe()
print(desc)

print("\nmatches describe()?")
print("mean:  ", np.isclose(manual_mean, desc["mean"]))
print("std:   ", np.isclose(manual_std, desc["std"]))
print("median:", np.isclose(manual_median, desc["50%"]))
print("var:   ", np.isclose(manual_var, age.var()))  # pandas .var() also defaults to ddof=1

In [ ]:
# Computing quantiles with np.percentile()

pre_25 = np.percentile(titanic["age"].dropna(), 25)  # 1st quartile
pre_50 = np.percentile(titanic["age"].dropna(), 50)  # Median
pre_75 = np.percentile(titanic["age"].dropna(), 75)  # 3rd quartile


print("\nQuantiles:")
print("25th percentile:", pre_25)
print("50th percentile:", pre_50)
print("75th percentile:", pre_75)

In [ ]:
# Compute with df quantiles = titanic["age"].quantile([0.25, 0.5, 0.75])

titanic_quantiles = titanic["age"].quantile([0.25, 0.5, 0.75])
print("\nQuantiles (pandas):")
print(titanic_quantiles)

### IQR outlier check on `fare`

Standard rule: anything outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` is flagged as an outlier.

In [ ]:
fare = titanic["fare"]

q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("bounds:", lower_bound, "to", upper_bound)

# convert to boolean mask for outliers
mask = (fare < lower_bound) | (fare > upper_bound)

# get the outliers
outliers = titanic[mask]
print("\nnumber of outliers:", len(outliers))
print("share of dataset:", len(outliers) / len(titanic))

In [ ]:
outliers[["fare", "pclass", "class", "survived"]].sort_values("fare", ascending=False).head(10)

### Visualizing it: boxplot of `fare`

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
sns.boxplot(x=titanic["fare"], showmeans=True)
plt.axvline(upper_bound, color="red", linestyle="--", label=f"upper fence ({upper_bound:.1f})")
plt.legend()
plt.title("Fare distribution — box = Q1 to Q3, whisker = 1.5*IQR fence, dots = outliers, triangle = mean")
plt.show()

In [ ]:
# Group BY

In [ ]:
titanic.groupby('pclass')['fare'].mean()

titanic.groupby('pclass').get_group(1)

In [ ]:
titanic.groupby("sex")["survived"].mean()

In [ ]:
# Given a passenger survived , 
# what is the probability that the passenger is a female?

In [ ]:
mask = titanic["survived"] == 1
survived_passengers = titanic[mask]

female_survivors = survived_passengers[survived_passengers["sex"] == "female"]
ans = len(female_survivors) / len(survived_passengers)

print("Probability that a passenger is female given that they survived:", ans)

In [ ]:
# Using Bayes theorem
#  A -> Survived
#  B -> Female

# P(B|A) = P(A|B) * P(B) / P(A)

# P(A|B) = Probability that a passenger survived given that they are female

mask_female = titanic["sex"] == "female"
female_passengers = titanic[mask_female]
p_a_given_b = len(female_passengers[female_passengers["survived"] == 1]) / len(female_passengers)

p_female = len(female_passengers) / len(titanic)

p_survived = len(survived_passengers) / len(titanic)

p_b_given_a = p_a_given_b * p_female / p_survived

print("Probability that a passenger survived given that they are female:", p_b_given_a)

In [ ]:
# Distribution curve for fare
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
sns.histplot(titanic["fare"], kde=True, bins=60)
plt.axvline(titanic["fare"].mean(), color="green", linestyle="--", label=f"mean ({titanic['fare'].mean():.1f})")
plt.axvline(titanic["fare"].median(), color="orange", linestyle="--", label=f"median ({titanic['fare'].median():.1f})")
plt.legend()
plt.title("Fare distribution — histogram + KDE curve")
plt.show()